# TEACHER COPY - Practical 3: Simple A* (grid + 8-puzzle)

> This copy shows the expected outputs already filled in. Students run the clean notebook themselves. There is no code for students to write; this lab is run-and-observe.

---

# Practical 3 - Smart Search with A* 🤖⭐

### Artificial Intelligence (01AI0511) · Unit 2 · Samatrix.io

Today we watch a computer find the **shortest path** using a smart search called **A\*** (say "A-star").

A\* uses a **heuristic**: a quick **guess** of how far the goal still is.
We will try a **good guess** and a **no guess**, and see which one makes the computer
search **less**. That is the big lesson of today: **a good heuristic saves work.**

**You do not type any code today.** Just run each grey box from top to bottom and read what
happens. Every line of code is written in a simple way, so you can read it too.

---
### How to run a box
Click a grey code box, then press **Shift + Enter**. Run the boxes **top to bottom**, one by one.


## Step 1 - Wake up 👋

Run the box below. You should see a friendly message.


In [1]:
print("Hello! I am your smart-search helper. 🤖")
print("Give me a puzzle and I will find the shortest answer with A*! ⭐")


Hello! I am your smart-search helper. 🤖
Give me a puzzle and I will find the shortest answer with A*! ⭐


Saw the message? 🎉 **Python works. Let's build A\*.**

## Step 2 - The A* search (read it, then run it) 🧠

Here is A\* written in the **simplest way**, using only a plain **list**. No tricky tools.

The idea:
- We keep a **to-do list** of places to explore. Each item on the list is a small note:
  `[f, g, place, path]`.
  - **g** = steps taken so far to reach this place.
  - **h** = the **guess** of steps still left (the heuristic).
  - **f = g + h** = the total. A\* always explores the **smallest f** first.
- We look through the list, pick the note with the smallest `f`, and explore it.
- We keep a list of places we already **visited**, so we do not repeat them.
- We count how many places we explore, so we can **compare heuristics** later.

You do not need to change anything. Just read the comments and run it.


In [2]:
def astar(start, goal, get_neighbours, guess):
    # The to-do list. Each note is [f, g, place, path_so_far].
    frontier = [[guess(start), 0, start, [start]]]
    visited = []            # places we already explored
    explored_count = 0      # how many places we explored (for comparing)

    while len(frontier) > 0:
        # 1) Find the note with the SMALLEST f by looking through the list.
        best = 0
        for i in range(len(frontier)):
            if frontier[i][0] < frontier[best][0]:
                best = i
        note = frontier.pop(best)     # take that note out of the list

        f, g, place, path = note      # unpack the note

        # 2) If we already explored this place, skip it.
        if place in visited:
            continue
        visited.append(place)
        explored_count = explored_count + 1

        # 3) Did we reach the goal? Then we are done.
        if place == goal:
            return path, explored_count

        # 4) Add each neighbour to the to-do list as a new note.
        for nxt in get_neighbours(place):
            if nxt not in visited:
                new_g = g + 1                       # one more step
                new_f = new_g + guess(nxt)          # f = g + h
                frontier.append([new_f, new_g, nxt, path + [nxt]])

    return None, explored_count     # no path found

print("A* is ready. ✅")


A* is ready. ✅


That is the whole search. Notice there is **nothing fancy**: just a list, a loop,
and 'pick the smallest f'. Now let's give it a puzzle to solve.


# Part A - A Robot on a Grid 🗺️

First, an easy picture: a robot on a small grid must reach the goal.


## Step 3 - Build the grid

Each box is one step.
- `S` = Start, `G` = Goal 🎯
- `#` = Wall (cannot pass), `.` = open floor

Just run it. It draws the grid.


In [3]:
grid = [
    "...........",
    "....###....",
    "S.........G",
    "....###....",
    "...........",
]

def find(letter):
    for r in range(len(grid)):
        for c in range(len(grid[r])):
            if grid[r][c] == letter:
                return (r, c)

start = find("S")
goal  = find("G")
rows, cols = len(grid), len(grid[0])

print("Start S is at", start)
print("Goal  G is at", goal)
print()
for line in grid:
    print("  " + "  ".join(line))


Start S is at (2, 0)
Goal  G is at (2, 10)

  .  .  .  .  .  .  .  .  .  .  .
  .  .  .  .  #  #  #  .  .  .  .
  S  .  .  .  .  .  .  .  .  .  G
  .  .  .  .  #  #  #  .  .  .  .
  .  .  .  .  .  .  .  .  .  .  .


## Step 4 - Where can the robot step?

From any box, the robot can move up, down, left, or right, if it stays on the grid
and does not hit a wall. Run it.


In [4]:
def grid_neighbours(box):
    r, c = box
    out = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:   # up, down, left, right
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:           # stay on the grid
            if grid[nr][nc] != "#":                     # not a wall
                out.append((nr, nc))
    return out

print("From the start", start, "the robot can step to:", grid_neighbours(start))


From the start (2, 0) the robot can step to: [(1, 0), (3, 0), (2, 1)]


## Step 5 - Two guesses (heuristics)

- **Zero guess**: always guess `0`. No real hint, so A\* spreads out everywhere.
- **Manhattan guess**: count steps up/down **plus** steps left/right to the goal.
  A smart hint, so A\* heads toward the goal.

Run it. Both are short and simple.


In [5]:
# Guess 1: no hint at all
def grid_zero(box):
    return 0

# Guess 2: the Manhattan distance (rows apart + columns apart)
def grid_manhattan(box):
    r, c = box
    return abs(r - goal[0]) + abs(c - goal[1])

print("Manhattan guess from the start:", grid_manhattan(start))


Manhattan guess from the start: 10


## Step 6 - Solve the grid and compare 🔬

Now we run A\* with **each** guess and compare. Watch the **explored** numbers.


In [6]:
path_zero, explored_zero = astar(start, goal, grid_neighbours, grid_zero)
path_manh, explored_manh = astar(start, goal, grid_neighbours, grid_manhattan)

print("Guess          Path length    Boxes explored")
print("-" * 46)
print("Zero (no hint) ", len(path_zero) - 1, "steps      ", explored_zero)
print("Manhattan      ", len(path_manh) - 1, "steps      ", explored_manh)
print()
print("Same shortest path. But Manhattan explores FEWER boxes. ⭐")


Guess          Path length    Boxes explored
----------------------------------------------
Zero (no hint)  10 steps       43
Manhattan       10 steps       11

Same shortest path. But Manhattan explores FEWER boxes. ⭐


## Step 7 - See the path 👀

Let's draw the robot's route with stars `*`.


In [7]:
def show_grid_path(path):
    path_set = set(path)
    for r in range(rows):
        line = ""
        for c in range(cols):
            if (r, c) == start:
                line += " S "
            elif (r, c) == goal:
                line += " G "
            elif (r, c) in path_set:
                line += " * "
            else:
                line += " " + grid[r][c] + " "
        print(line)

print("The robot's path:")
print()
show_grid_path(path_manh)


The robot's path:

 .  .  .  .  .  .  .  .  .  .  . 
 .  .  .  .  #  #  #  .  .  .  . 
 S  *  *  *  *  *  *  *  *  *  G 
 .  .  .  .  #  #  #  .  .  .  . 
 .  .  .  .  .  .  .  .  .  .  . 


# Part B - The 8-Puzzle 🧩

Now the **same A\*** solves a different puzzle: the sliding 8-puzzle.

There are 8 numbered tiles and one empty space in a 3 by 3 box.
You slide a tile into the empty space. The goal is to get the numbers in order,
with the empty space last:

```
1 2 3
4 5 6
7 8 _
```

We write each arrangement as a short text of 9 characters, where `0` is the empty space.
So the goal looks like `"123456780"`.


## Step 8 - Set up the puzzle

`start_state` is a scrambled puzzle. Run it to see the start and the goal.


In [8]:
goal_state  = "123456780"     # the tidy goal (0 is the empty space)
start_state = "123450678"     # a scrambled start

def show_puzzle(state):
    for r in range(3):
        row = state[r*3 : r*3 + 3]
        row = row.replace("0", "_")          # show the empty space as _
        print("   " + "  ".join(row))

print("Start:")
show_puzzle(start_state)
print()
print("Goal:")
show_puzzle(goal_state)


Start:
   1  2  3
   4  5  _
   6  7  8

Goal:
   1  2  3
   4  5  6
   7  8  _


## Step 9 - How the tiles can slide

From any arrangement, we can slide the empty space up, down, left, or right.
Each slide gives a new arrangement. Run it to see the arrangements we can reach
from the start.


In [9]:
def puzzle_neighbours(state):
    out = []
    zero = state.index("0")          # where the empty space is
    r, c = zero // 3, zero % 3
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:          # stay inside the box
            new_zero = nr * 3 + nc
            tiles = list(state)
            # swap the empty space with the tile next to it
            tiles[zero], tiles[new_zero] = tiles[new_zero], tiles[zero]
            out.append("".join(tiles))
    return out

print("From the start, we can reach these arrangements:")
for s in puzzle_neighbours(start_state):
    print("  ", s)


From the start, we can reach these arrangements:
   120453678
   123458670
   123405678


## Step 10 - Two guesses for the puzzle

- **Zero guess**: always `0` (no hint).
- **Wrong-tiles guess**: count how many tiles are **not** in their goal spot.
  Fewer wrong tiles means we are closer, so this is a smart hint.

Run it.


In [10]:
# Guess 1: no hint
def puzzle_zero(state):
    return 0

# Guess 2: count tiles that are NOT in the right place (ignore the empty space)
def puzzle_wrong_tiles(state):
    count = 0
    for i in range(9):
        if state[i] != "0" and state[i] != goal_state[i]:
            count = count + 1
    return count

print("Wrong tiles in the start:", puzzle_wrong_tiles(start_state))


Wrong tiles in the start: 3


## Step 11 - Solve the puzzle and compare 🔬

Here is the key experiment. We solve the **same puzzle** with each guess and compare
how many arrangements A\* had to explore.


In [11]:
ans_zero,  count_zero  = astar(start_state, goal_state, puzzle_neighbours, puzzle_zero)
ans_smart, count_smart = astar(start_state, goal_state, puzzle_neighbours, puzzle_wrong_tiles)

print("Guess            Moves to solve    Arrangements explored")
print("-" * 56)
print("Zero (no hint)  ", len(ans_zero) - 1,  "moves          ", count_zero)
print("Wrong-tiles     ", len(ans_smart) - 1, "moves          ", count_smart)
print()
print("Same number of moves. But the smart guess explores FAR fewer arrangements! ⭐")


Guess            Moves to solve    Arrangements explored
--------------------------------------------------------
Zero (no hint)   13 moves           2710
Wrong-tiles      13 moves           190

Same number of moves. But the smart guess explores FAR fewer arrangements! ⭐


## Step 12 - Watch the puzzle get solved 🎬

Let's print the arrangements from start to goal, step by step.


In [12]:
print("Solving the puzzle, one slide at a time:")
print()
for step, state in enumerate(ans_smart):
    print("Move", step, ":")
    show_puzzle(state)
    print()


Solving the puzzle, one slide at a time:

Move 0 :
   1  2  3
   4  5  _
   6  7  8

Move 1 :
   1  2  3
   4  5  8
   6  7  _

Move 2 :
   1  2  3
   4  5  8
   6  _  7

Move 3 :
   1  2  3
   4  5  8
   _  6  7

Move 4 :
   1  2  3
   _  5  8
   4  6  7

Move 5 :
   1  2  3
   5  _  8
   4  6  7

Move 6 :
   1  2  3
   5  6  8
   4  _  7

Move 7 :
   1  2  3
   5  6  8
   4  7  _

Move 8 :
   1  2  3
   5  6  _
   4  7  8

Move 9 :
   1  2  3
   5  _  6
   4  7  8

Move 10 :
   1  2  3
   _  5  6
   4  7  8

Move 11 :
   1  2  3
   4  5  6
   _  7  8

Move 12 :
   1  2  3
   4  5  6
   7  _  8

Move 13 :
   1  2  3
   4  5  6
   7  8  _



## Step 13 - The Big Lesson 🎯

Look back at the two experiments (Step 6 and Step 11).

| Puzzle | No-hint guess explores | Smart guess explores |
|--------|------------------------|----------------------|
| Grid   | many boxes             | few boxes            |
| 8-puzzle | very many arrangements | far fewer          |

In both cases:
- The answer (path length or number of moves) is the **same**.
- But a **smart heuristic explores much less**, so A\* is faster.

**That is the whole point: a good heuristic guides the search and saves work.** 🎉


## 🏆 Well Done!

Today you saw:

- 🤖 A\* finds the **shortest answer**, using a to-do list and picking the smallest **f = g + h**.
- 🧭 A **heuristic** is a quick **guess** of how far the goal is.
- 🗺️ On a **grid**, the Manhattan guess beats no guess.
- 🧩 On the **8-puzzle**, the wrong-tiles guess beats no guess.
- ⭐ A **good heuristic explores fewer states** but finds the same best answer.

You did not change a single line, but you saw A\* and heuristics in action. Great job! 🎉🤖⭐

---
*Artificial Intelligence (01AI0511) · Unit 2 · Practical 3 · Samatrix.io*
